In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import pearsonr
from scipy.stats import spearmanr
from scipy.stats import mannwhitneyu

import sys
sys.path.append('/ems/elsc-labs/habib-n/yuval.rom/BCG/biomarkers/')
from tau217.utils.utils import *
sys.path.append('/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/clinical_data_analysis/statistical_analysis/')
from settings import *


In [ ]:
FIG_DIR = '/ems/elsc-labs/habib-n/yuval.rom/BCG/biomarkers/final_figures'
SEX_FIG_DIR = '/ems/elsc-labs/habib-n/yuval.rom/BCG/biomarkers/sex_figures'
APOE_FIG_DIR = '/ems/elsc-labs/habib-n/yuval.rom/BCG/biomarkers/APOE_figures'

In [ ]:
data_path = '/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/Data/fightAD_general_data_table.xlsx'
df = pd.read_excel(data_path, sheet_name='1')
df.columns = df.loc[0]
df = df.drop(0)
df = df.iloc[:,:-2]
print(df.shape)
df.head(1)

In [ ]:
df.rename(columns={'#':'ID',
                   'תאריך בדיקות דם': 'blood_test_date',
                   'שעת לקיחת דם':'blood_taken_time',
                   'מספר מנות':'num_of_injections',
                   'גיל הפסקת מחזור':'menopause',
                   'אסטרוגן':'estorgen',
                   'Frontal memory score (#1)- כמה מילים זכר/ה ברמז הראשון. נקודה אם זכר/ה רמז ראשון ושתי נקודות אם לא.':'Frontal memory score (#1)',
                   'Hipocampal memory score (#2)- כמה מילים לא זכר/ה בכלל':'Hipocampal memory score (#2)',
                   'p-tau_ new_#1':'p-tau_new_#1',
                   'ביקורת אחרי שנה':'checkup after a year',
                   'חיסון':'vaccine',
                   'תרופות':'medications',
                   'הערות':'notes',
                   'Number_of_risk_faktors':'Number_of_risk_factors',
                   'P-tau217':'p-tau217',
                   'צרבונין':'Cerebonin',
                   }, inplace=True)

In [ ]:
df = df.apply(pd.to_numeric, errors='ignore')

In [ ]:
# outliers = []  # Participant exclusions are maintained privately.

print(df.shape) 
print(df['GFAP'].describe())
df = df.drop(df[df['ID'].isin(OUTLIERS)].index)
print(df['GFAP'].describe())
print(df.shape) 

In [ ]:
# df = df[df[TAU_217].notna()] # Filter only to those with tau_217 values
df = df[df[MOCA_COL].notna()] # Filter only to those with MOCA values
df = df[df[MOCA_COL] != '?']
df['MCI'] = df[MOCA_COL] < MCI_THR
df['Negative_MOCA'] = df[MOCA_COL] * -1
df[TAU_217_CAT] = pd.cut(df[TAU_217], bins=[0, INT_LOW_TAU_217_THR, INT_HIGH_TAU_217_THR, HIGH_TAU_217_THR, np.inf], labels=[0, 1, 2, 3], right=False)

df_high_vs_low = df[(df[TAU_217] > HIGH_TAU_217_THR) | (df[TAU_217] < INT_LOW_TAU_217_THR)]
df_high_vs_low[TAU_217_CAT] = df_high_vs_low[TAU_217_CAT].cat.remove_categories([1])

df_high_vs_low.groupby(TAU_217_CAT)['Negative_MOCA'].mean()

In [ ]:
df['GFAP'].plot(kind='hist', bins=100)
plt.title('GFAP')
df.iloc[df['GFAP'].argmax()].ID


In [ ]:
df['NFL'].plot(kind='hist', bins=100, label='NFL')
plt.title('NFL')
df.iloc[df['NFL'].argmax()].ID

In [ ]:
df[TAU_217].plot(kind='hist', bins=100, label=TAU_217)
plt.title(TAU_217)
df.iloc[df[TAU_217].argmax()].ID

In [ ]:
df['APOE'] = df['APOE-SNP'].replace({'APOE3/3': 0, 'APO3/3': 0, 'APOE3/4': 1, 'APO3/4': 1, 'APOE4/4': 2, 'APO4/4': 2})
df['APOE'].value_counts()

In [ ]:
biomarkeres_df = df[['ID', 'NFL', 'GFAP', TAU_217, TAU_217_CAT, 'Negative_MOCA', 'APOE']]
biomarkeres_df = df[['ID', 'NFL', 'GFAP', TAU_217, TAU_217_CAT, 'Negative_MOCA']]
# biomarkeres_df = df[['ID', 'NFL', 'GFAP', TAU_217,  TAU_217_CAT, 'Negative_MOCA', 'MCI', 'APOE']]
print(biomarkeres_df.shape)
# biomarkeres_df = biomarkeres_df.dropna()
biomarkeres_df = biomarkeres_df.set_index('ID')
biomarkeres_df = biomarkeres_df.dropna(subset=['NFL'])


print(biomarkeres_df.shape)
corr = biomarkeres_df.corr()
corr

# Scaling the biomarkers to the same range for comparision

In [ ]:
scaled_b_df = biomarkeres_df.copy()
scaler = MinMaxScaler()
scaled_b_df = pd.DataFrame(scaler.fit_transform(scaled_b_df), columns=scaled_b_df.columns, index=scaled_b_df.index)

In [ ]:
cols = ['NFL', 'GFAP', 'p-tau217', 'Negative_MOCA']
corr_matrix = scaled_b_df[cols].corr()
pval_matrix = pd.DataFrame(np.ones(corr_matrix.shape), columns=cols, index=cols)

for i in range(len(cols)):
    for j in range(len(cols)):
        if i != j:
            r, p = pearsonr(scaled_b_df[cols[i]], scaled_b_df[cols[j]])
            pval_matrix.iloc[i, j] = p
        else:
            pval_matrix.iloc[i, j] = np.nan

annot_matrix = corr_matrix.round(2).astype(str)
for i in range(len(cols)):
    for j in range(len(cols)):
        p = pval_matrix.iloc[i, j]
        if np.isnan(p):
            annot = ''
        elif p < 0.005:
            annot = f"{corr_matrix.iloc[i, j]:.2f}\n***"
        elif p < 0.01:
            annot = f"{corr_matrix.iloc[i, j]:.2f}\n**"
        elif p < 0.05:
            annot = f"{corr_matrix.iloc[i, j]:.2f}\n*"
        else:
            annot = f"{corr_matrix.iloc[i, j]:.2f}"
        annot_matrix.iloc[i, j] = annot

In [ ]:
sns.heatmap(scaled_b_df[cols].corr(), annot=annot_matrix, vmin=0, vmax=1, fmt='')
plt.title(f'Pearson correlation heatmap N={scaled_b_df.shape[0]}')
plt.savefig(f'{FIG_DIR}/pearson_correlation_heatmap.pdf', bbox_inches='tight', dpi=300, format='pdf')

In [ ]:
# Compute Spearman correlation and p-value matrices
spearman_corr, spearman_pval = spearmanr(scaled_b_df[cols])
spearman_corr_matrix = pd.DataFrame(spearman_corr, columns=cols, index=cols)
spearman_pval_matrix = pd.DataFrame(spearman_pval, columns=cols, index=cols)

# Create annotation matrix for Spearman
spearman_annot_matrix = spearman_corr_matrix.round(2).astype(str)
for i in range(len(cols)):
    for j in range(len(cols)):
        p = spearman_pval_matrix.iloc[i, j]
        if np.isnan(p):
            annot = ''
        elif p < 0.005:
            annot = f"{spearman_corr_matrix.iloc[i, j]:.2f}\n***"
        elif p < 0.01:
            annot = f"{spearman_corr_matrix.iloc[i, j]:.2f}\n**"
        elif p < 0.05:
            annot = f"{spearman_corr_matrix.iloc[i, j]:.2f}\n*"
        else:
            annot = f"{spearman_corr_matrix.iloc[i, j]:.2f}"
        spearman_annot_matrix.iloc[i, j] = annot

sns.heatmap(spearman_corr_matrix, annot=spearman_annot_matrix, vmin=0, vmax=1, fmt='')
plt.title(f'Spearman correlation heatmap N={scaled_b_df.shape[0]}')
plt.savefig(f'{FIG_DIR}/spearman_correlation_heatmap.pdf', dpi=300, format='pdf')
plt.show()


In [ ]:
# # Compute Spearman correlation and p-value matrices
# # filter for only high p-tau217 samples
# # from BCG.biomarkers.tau217.feature_disterbution import SCALED_TAU_217_THR


# scaled_b_df = scaled_b_df[scaled_b_df[TAU_217] > SCALED_HIGH_TAU_217_THR]
# spearman_corr, spearman_pval = spearmanr(scaled_b_df[cols])
# spearman_corr_matrix = pd.DataFrame(spearman_corr, columns=cols, index=cols)
# spearman_pval_matrix = pd.DataFrame(spearman_pval, columns=cols, index=cols)

# # Create annotation matrix for Spearman
# spearman_annot_matrix = spearman_corr_matrix.round(2).astype(str)
# for i in range(len(cols)):
#     for j in range(len(cols)):
#         p = spearman_pval_matrix.iloc[i, j]
#         if np.isnan(p):
#             annot = ''
#         elif p < 0.005:
#             annot = f"{spearman_corr_matrix.iloc[i, j]:.2f}\n***"
#         elif p < 0.01:
#             annot = f"{spearman_corr_matrix.iloc[i, j]:.2f}\n**"
#         elif p < 0.05:
#             annot = f"{spearman_corr_matrix.iloc[i, j]:.2f}\n*"
#         else:
#             annot = f"{spearman_corr_matrix.iloc[i, j]:.2f}"
#         spearman_annot_matrix.iloc[i, j] = annot

# sns.heatmap(spearman_corr_matrix, annot=spearman_annot_matrix, vmin=0, vmax=1, fmt='')
# plt.title(f'Spearman correlation heatmap N={scaled_b_df.shape[0]}')
# plt.savefig('final_figures/spearman_correlation_heatmap.pdf')

In [ ]:
dummy_array = np.zeros((1, len(scaler.scale_)))
scaled_thresholds = {}
feature_indices = {name: idx for idx, name in enumerate(scaler.feature_names_in_)}
for threshold, feature in zip([HIGH_TAU_217_THR], [TAU_217]):
    # Get the index for this feature
    idx = feature_indices[feature]
    
    # Create a copy of the dummy array
    threshold_array = dummy_array.copy()
    threshold_array[0, idx] = threshold
    
    # Transform the array and get the scaled threshold
    scaled_value = scaler.transform(threshold_array)[0, idx]
    scaled_thresholds[feature] = scaled_value

# scaled_thresholds
SCALED_HIGH_TAU_217_THR = scaled_thresholds[TAU_217]


In [ ]:
sns.kdeplot(scaled_b_df['GFAP'], color='r', alpha=0.5, label='GFAP', fill=True)
sns.kdeplot(scaled_b_df['NFL'], color='g', alpha=0.5, label='NFL', fill=True)
sns.kdeplot(scaled_b_df[TAU_217], color='b', alpha=0.5, label=TAU_217, fill=True)

plt.title('Biomarkers Density Plot')
plt.xlabel('Scaled Values')
plt.ylabel('Density')
plt.legend()
# plt.savefig('final_figures/biomarkers_density_plot.pdf')

In [ ]:
n_bins = 30
sns.histplot(scaled_b_df['GFAP'], color='g', alpha=0.5, label='GFAP', fill=True, bins=n_bins, kde=True)
sns.histplot(scaled_b_df['NFL'], color='r', alpha=0.5, label='NFL', fill=True, bins=n_bins, kde=True)
sns.histplot(scaled_b_df[TAU_217], color='b', alpha=0.5, label=TAU_217, fill=True, bins=n_bins, kde=True)

plt.title('Biomarkers Histogram Plot')
plt.xlabel('Scaled Values')
plt.ylabel('Count')
plt.legend()
plt.savefig(f'{FIG_DIR}/biomarkers_histograms_plot.pdf', bbox_inches='tight', dpi=300, format='pdf')

### Distribution of biomarkers in the presence of APOE4

In [ ]:
# p-tau217 distribution in with different APOE genotypes
df['APOE'] = df['APOE-SNP'].replace({'APOE3/3': 0, 'APO3/3': 0, 'APOE3/4': 1, 'APO3/4': 1, 'APOE4/4': 2, 'APO4/4': 2})
df['APOE'].value_counts()

In [ ]:
compare_groups_statistical(
    df=df, 
    group_col='APOE', 
    feature_col=TAU_217,
    group1_filter=0,  # APOE3/3
    group2_filter=lambda x: (x == 1) | (x == 2),  # APOE3/4 or APOE4/4
    group1_name='APOE3/3',
    group2_name='APOE3/4+4/4',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='p-tau217',
    fig_dir=APOE_FIG_DIR
)


In [ ]:


# plt.ylabel('Density')
# plt.legend()
# plt.show()

compare_groups_statistical(
    df=df, 
    group_col='APOE', 
    group1_filter=0,  # APOE3/3
    group2_filter=lambda x: (x == 1) | (x == 2),  # APOE3/4 or APOE4/4
    group1_name='APOE3/3',
    group2_name='APOE3/4+4/4',
    test_type='mannwhitneyu',
    alternative='greater',
    fig_dir=APOE_FIG_DIR
)

In [ ]:
# # NFL distribution in with different APOE genotypes
# sns.histplot(df_apoe_0['NFL'], color='b', alpha=0.5, label=f'APOE3/3 N={df_apoe_0.shape[0]}', kde=True, stat='density', bins=30)
# sns.histplot(df_apoe_1['NFL'], color='r', alpha=0.5, label=f'APOE3/4 N={(df['APOE'] == 1).sum()} + APOE4/4 N={(df['APOE'] == 2).sum()}', kde=True, stat='density', bins=30)

# plt.title('NFL Distribution by APOE Genotype')
# plt.xlabel('NFL')
# plt.ylabel('Density')
# plt.legend()
# plt.show()
compare_groups_statistical(
    df=df, 
    group_col='APOE', 
    feature_col=NFL,
    group1_filter=0,  # APOE3/3
    group2_filter=lambda x: (x == 1) | (x == 2),  # APOE3/4 or APOE4/4
    group1_name='APOE3/3',
    group2_name='APOE3/4+4/4',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='NFL',
    fig_dir=APOE_FIG_DIR
)


In [ ]:
compare_groups_statistical(
    df=df, 
    group_col='APOE', 
    feature_col=GFAP,
    group1_filter=0,  # APOE3/3
    group2_filter=lambda x: (x == 1) | (x == 2),  # APOE3/4 or APOE4/4
    group1_name='APOE3/3',
    group2_name='APOE3/4+4/4',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='GFAP',
    fig_dir=APOE_FIG_DIR
)

### High vs. low p-tau217 individuals GFAP and NFL

In [ ]:
df_217 = df[df[TAU_217].notna()]
df_217[TAU_217_CAT] = pd.cut(df_217[TAU_217], bins=[0, INT_LOW_TAU_217_THR, INT_HIGH_TAU_217_THR, HIGH_TAU_217_THR, np.inf], labels=['Low', 'Intermediate_low', 'Intermediate_high', 'High'])
df_217 = df_217[df_217[TAU_217].notna()]

df_high_vs_low = df_217[(df_217[TAU_217] > HIGH_TAU_217_THR) | (df_217[TAU_217] < INT_LOW_TAU_217_THR)]
df_high_vs_low[TAU_217_CAT] = df_high_vs_low[TAU_217_CAT].cat.remove_categories(['Intermediate_low', 'Intermediate_high'])

In [ ]:
scaled_b_df.describe()

In [ ]:
compare_groups_statistical(
    df=df_high_vs_low, 
    group_col=TAU_217_CAT, 
    feature_col=GFAP,
    group1_filter=lambda x: (x == "Low"),
    group2_filter=lambda x: (x == "High"), 
    group1_name='Low p-tau217',
    group2_name='High p-tau217',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='GFAP',
    fig_dir=FIG_DIR
)

In [ ]:
compare_groups_statistical(
    df=df_high_vs_low, 
    group_col=TAU_217_CAT, 
    feature_col=NFL,
    group1_filter=lambda x: (x == "Low"),
    group2_filter=lambda x: (x == "High"), 
    group1_name='Low p-tau217',
    group2_name='High p-tau217',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='NFL',
    fig_dir=FIG_DIR
)

### GFAP and NFL distribution in MCI and cognitive healthy individuals

In [ ]:
nci_df = df_217[df_217['MCI'] == 0]
mci_df = df_217[df_217['MCI'] == 1]

In [ ]:
compare_groups_statistical(
    df=df, 
    group_col='MCI', 
    feature_col=GFAP,
    group1_filter=lambda x: (x == 0),
    group2_filter=lambda x: (x == 1), 
    group1_name='Non-MCI',
    group2_name='MCI',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='GFAP',
    fig_dir=FIG_DIR
)

In [ ]:
compare_groups_statistical(
    df=df, 
    group_col='MCI', 
    feature_col=NFL,
    group1_filter=lambda x: (x == 0),
    group2_filter=lambda x: (x == 1), 
    group1_name='Non-MCI',
    group2_name='MCI',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='NFL',
    fig_dir=FIG_DIR
)

### AD-MCI vs Non-AD MCI

In [ ]:
mci_df = df_217[df_217['MCI'] == 1]

In [ ]:
compare_groups_statistical(
    df=mci_df, 
    group_col=TAU_217_CAT, 
    feature_col=TAU_217,
    group1_filter=lambda x: (x == 'Low'),
    group2_filter=lambda x: (x == 'High'), 
    group1_name='non-AD MCI',
    group2_name='AD-MCI',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name=TAU_217,
    fig_dir=FIG_DIR
)

In [ ]:


compare_groups_statistical(
    df=mci_df, 
    group_col=TAU_217_CAT, 
    feature_col=NFL,
    group1_filter=lambda x: (x == 'Low'),
    group2_filter=lambda x: (x == 'High'), 
    group1_name='non-AD MCI',
    group2_name='AD-MCI',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='NFL',
    fig_dir=FIG_DIR
)

In [ ]:
compare_groups_statistical(
    df=mci_df, 
    group_col=TAU_217_CAT, 
    feature_col=GFAP,
    group1_filter=lambda x: (x == 'Low'),
    group2_filter=lambda x: (x == 'High'), 
    group1_name='non-AD MCI',
    group2_name='AD-MCI',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='GFAP',
    fig_dir=FIG_DIR
)

### Cognitively Healthy

In [ ]:
nci_df = df_217[df_217['MCI'] == 0]

compare_groups_statistical(
    df=nci_df, 
    group_col=TAU_217_CAT, 
    feature_col=GFAP,
    group1_filter=lambda x: (x == 'Low'),
    group2_filter=lambda x: (x == 'High'), 
    group1_name='Cognitively healthy p-tau217 low',
    group2_name='Cognitively healthy p-tau217 high',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='GFAP',
    fig_dir=FIG_DIR
)

In [ ]:
compare_groups_statistical(
    df=nci_df, 
    group_col=TAU_217_CAT, 
    feature_col=NFL,
    group1_filter=lambda x: (x == 'Low'),
    group2_filter=lambda x: (x == 'High'), 
    group1_name='Cognitively healthy p-tau217 low',
    group2_name='Cognitively healthy p-tau217 high',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='NFL',
    fig_dir=FIG_DIR
)

In [ ]:
nci_df_181 = df[df['MCI'] == 0]

compare_groups_statistical(
    df=nci_df_181, 
    feature_col=GFAP,
    group1_filter=lambda x: (x == 'Low'),
    group2_filter=lambda x: (x == 'High'), 
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='GFAP',
    fig_dir=FIG_DIR
)

In [ ]:
compare_groups_statistical(
    df=nci_df, 
    group_col=TAU_217_CAT, 
    feature_col=NFL,
    group1_filter=lambda x: (x == 'Low'),
    group2_filter=lambda x: (x == 'High'), 
    group1_name='Cognitively healthy p-tau217 low',
    group2_name='Cognitively healthy p-tau217 high',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='NFL',
    fig_dir=FIG_DIR
)

### APOE

In [ ]:
compare_groups_statistical(
    df=nci_df, 
    group_col='APOE', 
    feature_col=GFAP,
    group1_filter=0,  # APOE3/3
    group2_filter=lambda x: (x == 1) | (x == 2),  # APOE3/4 or APOE4/4
    group1_name='APOE3/3',
    group2_name='APOE3/4+4/4',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='p-tau217',
    fig_dir=APOE_FIG_DIR
)
compare_groups_statistical(
    df=nci_df, 
    group_col='APOE', 
    feature_col=GFAP,
    group1_filter=0,  # APOE3/3
    group2_filter=lambda x: (x == 1) | (x == 2),  # APOE3/4 or APOE4/4
    group1_name='APOE3/3',
    group2_name='APOE3/4+4/4',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='NFL',
    fig_dir=APOE_FIG_DIR
)
compare_groups_statistical(
    df=nci_df, 
    group_col='APOE', 
    feature_col=GFAP,
    group1_filter=0,  # APOE3/3
    group2_filter=lambda x: (x == 1) | (x == 2),  # APOE3/4 or APOE4/4
    group1_name='APOE3/3',
    group2_name='APOE3/4+4/4',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='GFAP',
    fig_dir=APOE_FIG_DIR
)


# Sex analysis

Are any of the biomarkers effected by sex

In [ ]:
compare_groups_statistical(
    df=df, 
    group_col='Gender', 
    feature_col=TAU_217,
    group1_filter=lambda x: (x == 'm'),  # APOE3/3
    group2_filter=lambda x: (x == 'f'),  # APOE3/4 or APOE4/4
    group1_name='Male',
    group2_name='Female',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='p-tau217',
    fig_dir=SEX_FIG_DIR
)

In [ ]:
compare_groups_statistical(
    df=df[df[TAU_217_CAT] == 3], 
    group_col='Gender', 
    feature_col=TAU_217,
    group1_filter=lambda x: (x == 'm'),  # APOE3/3
    group2_filter=lambda x: (x == 'f'),  # APOE3/4 or APOE4/4
    group1_name='Male',
    group2_name='Female',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='p-tau217',
    fig_dir=SEX_FIG_DIR
)

Same but for GFAP and NFL

In [ ]:
compare_groups_statistical(
    df=df, 
    group_col='Gender', 
    feature_col=GFAP,
    group1_filter=lambda x: (x == 'm'),  # APOE3/3
    group2_filter=lambda x: (x == 'f'),  # APOE3/4 or APOE4/4
    group1_name='Male',
    group2_name='Female',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='GFAP',
    fig_dir=SEX_FIG_DIR
)

In [ ]:
compare_groups_statistical(
    df=df[df[TAU_217_CAT] == 3], 
    group_col='Gender', 
    feature_col=GFAP,
    group1_filter=lambda x: (x == 'm'),  # APOE3/3
    group2_filter=lambda x: (x == 'f'),  # APOE3/4 or APOE4/4
    group1_name='Male',
    group2_name='Female',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='GFAP',
    fig_dir=SEX_FIG_DIR
)

Does females have higher tendency for vascular risks?

In [ ]:
hyperlipidemia_bool = (df['Hyperlipidemia'] == 'YES') | (df['Hyperlipidemia'] == 1)
df['Hyperlipidemia'] = hyperlipidemia_bool.astype(int)   
df['Triglycerides'].replace({'38..8': 38.8}, inplace=True)
df["non-hdl_cholesterol"] = df["Cholesterol"] - df["HDL"]

df['lipid_metaboloisim_disorder'] = (
    hyperlipidemia_bool | 
    (df['HDL'] < 45) | 
    (df['LDL'] > 100) | 
    (df['Triglycerides'] > 150)
)
df['uncontrolled_hyperlipidemia'] = (
    hyperlipidemia_bool 
    & (df['LDL'] > 116)
    & (df["non-hdl_cholesterol"] > 146)
)

In [ ]:
compare_groups_statistical(
    df=df, 
    group_col='uncontrolled_hyperlipidemia', 
    feature_col='p-tau217',
    group1_filter=lambda x: x == False,  # Not uncontrolled
    group2_filter=lambda x: x == True,   # Uncontrolled
    group1_name='uncontrolled_hyperlipidemia: False',
    group2_name='uncontrolled_hyperlipidemia: True',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='p-tau217',
    fig_dir=APOE_FIG_DIR
)
compare_groups_statistical(
    df=df, 
    group_col='uncontrolled_hyperlipidemia', 
    feature_col='NFL',
    group1_filter=lambda x: x == False,
    group2_filter=lambda x: x == True,
    group1_name='uncontrolled_hyperlipidemia: False',
    group2_name='uncontrolled_hyperlipidemia: True',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='NFL',
    fig_dir=APOE_FIG_DIR
)
compare_groups_statistical(
    df=df, 
    group_col='uncontrolled_hyperlipidemia', 
    feature_col='GFAP',
    group1_filter=lambda x: x == False,
    group2_filter=lambda x: x == True,
    group1_name='uncontrolled_hyperlipidemia: False',
    group2_name='uncontrolled_hyperlipidemia: True',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='GFAP',
    fig_dir=APOE_FIG_DIR
)


In [ ]:
df['Hyperlipidemia'].value_counts()
df['Gender'].value_counts()

In [ ]:
# Create a barplot of hyperlipidemia colored by Gender, normalized to number of samples of each sex
df['Gender'] = df['Gender'].replace({'F': 'f', 'M': 'm'})

# Calculate proportions of hyperlipidemia within each gender
hyperlipidemia_gender_counts = df.groupby(['Gender', 'Hyperlipidemia']).size().unstack(fill_value=0)
gender_totals = hyperlipidemia_gender_counts.sum(axis=1)
hyperlipidemia_gender_props = hyperlipidemia_gender_counts.div(gender_totals, axis=0)

# Prepare data for plotting
plot_df = hyperlipidemia_gender_props.reset_index().melt(id_vars='Gender', var_name='Hyperlipidemia', value_name='Proportion')

plt.figure(figsize=(6,4))
# Set custom palette: males ('m') blue, females ('f') red
custom_palette = {'m': 'blue', 'f': 'red'}
ax = sns.barplot(
    data=plot_df, 
    x='Hyperlipidemia', 
    y='Proportion', 
    hue='Gender', 
    palette=custom_palette, 
    alpha=0.7
)
ax.set_title('Proportion of Hyperlipidemia by Gender')
ax.set_xlabel('Hyperlipidemia')
ax.set_ylabel('Proportion within Gender')
plt.legend(title='Gender')
plt.tight_layout()
plt.savefig(SEX_FIG_DIR + 'hyperlipidemia_by_gender.png')
plt.show()


In [ ]:
compare_groups_statistical(
    df=df, 
    group_col='Gender', 
    feature_col='Triglycerides',
    group1_filter=lambda x: (x == 'm'),
    group2_filter=lambda x: (x == 'f'), 
    group1_name='Male',
    group2_name='Female',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='Triglycerides',
    fig_dir=SEX_FIG_DIR
)

compare_groups_statistical(
    df=df[df[TAU_217_CAT] == 3], 
    group_col='Gender', 
    feature_col='Triglycerides',
    group1_filter=lambda x: (x == 'm'),
    group2_filter=lambda x: (x == 'f'), 
    group1_name='Male',
    group2_name='Female',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='Triglycerides',
    fig_dir=SEX_FIG_DIR
)

In [ ]:
compare_groups_statistical(
    df=df, 
    group_col='Gender', 
    feature_col='LDL',
    group1_filter=lambda x: (x == 'm'),
    group2_filter=lambda x: (x == 'f'), 
    group1_name='Male',
    group2_name='Female',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='LDL',
    fig_dir=SEX_FIG_DIR
)

compare_groups_statistical(
    df=df[df[TAU_217_CAT] == 3], 
    group_col='Gender', 
    feature_col='LDL',
    group1_filter=lambda x: (x == 'm'),
    group2_filter=lambda x: (x == 'f'), 
    group1_name='Male',
    group2_name='Female',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='LDL',
    fig_dir=SEX_FIG_DIR
)

In [ ]:
compare_groups_statistical(
    df=df, 
    group_col='Gender', 
    feature_col='HDL',
    group1_filter=lambda x: (x == 'm'),
    group2_filter=lambda x: (x == 'f'), 
    group1_name='Male',
    group2_name='Female',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='HDL',
    fig_dir=SEX_FIG_DIR
)

compare_groups_statistical(
    df=df[df[TAU_217_CAT] == 3], 
    group_col='Gender', 
    feature_col='HDL',
    group1_filter=lambda x: (x == 'm'),
    group2_filter=lambda x: (x == 'f'), 
    group1_name='Male',
    group2_name='Female',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='HDL',
    fig_dir=SEX_FIG_DIR
)

### Checking if Diabetes that is controlled vs uncontrolled is a risk facotr for the biomarkeres

In [ ]:
def blood_tests_preprocess(df):
    df['HA1C_CAT'] = pd.cut(df['Hemoglobin_A1C'], bins=[0, HA1C_INT, HA1C_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])
    df['CHOL_CAT'] = pd.cut(df['Cholesterol'],  bins=[0, CHOL_INT, CHOL_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])
    df['Triglycerides'].replace({'38..8': 38.8}, inplace=True)
    df['TRG_CAT'] = pd.cut(df['Triglycerides'], bins=[0, TRG_INT, TRG_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])
    df['LDL_CAT'] = pd.cut(df['LDL'], bins=[0, LDL_INT, LDL_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])
    df['CRP'].replace({'<0.4': 0, '<0.5': 0, '<0.6':0}, inplace=True)
    df['SBP'] = df['BP'].str.split('/').str[0].astype(float)
    df['DBP'] = df['BP'].str.split('/').str[1].astype(float)
    # df = df[df['CRP'] != '<1']
    df['CRP'].replace({'<1': np.nan}, inplace=True)
    df['CRP_CAT'] = pd.cut(df['CRP'], bins=[0, CRP_INT, CRP_HIGH, CRP_VERY_HIGH, np.inf], labels=['Low', 'Intermediate', 'High', 'Very High'])
    df['WBC_CAT'] = pd.cut(df['WBC'], bins=[0, WBC_LOW, WBC_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])
    df['LYMPH_CAT'] = pd.cut(df['Lymphocytes'], bins=[0, LYMPH_LOW, LYMPH_HIGH, np.inf], labels=['Low', 'Intermediate', 'High'])

    return df

df = blood_tests_preprocess(df)

# Diabetes
print("Diabetes")
print((df['Diabetes'] == 'YES').sum())
print(df[(df['Hemoglobin_A1C'] >= 6.7)].shape[0])
# print(df[(df['Hemoglobin_A1C'] >= 6.1)].shape[0] / df[(df['Diabetes'] == 'YES')].shape[0])
print(df[(df['Diabetes'] == 'YES') & (df['Hemoglobin_A1C'] >= 6.7)].shape[0] / df[(df['Diabetes'] == 'YES')].shape[0])

In [ ]:
diabetes_not_controled = df[(df['Hemoglobin_A1C'] >= 6.5)]
diabetes_controled = df[(df['Hemoglobin_A1C'] < 6.5)]
df['Diabetes_controlled'] = np.nan
df['Diabetes_controlled'][(df['Hemoglobin_A1C'] < 6.5)] = 0
df['Diabetes_controlled'][(df['Hemoglobin_A1C'] >= 6.5)] = 1

all_diabetes = df[df['Diabetes'] == 'YES']
only_healthy = df[df['Diabetes'] == 'NO']
alpha = 0.2

compare_groups_statistical(
    df=df, 
    group_col='Diabetes_controlled', 
    feature_col=TAU_217,
    group1_filter=lambda x: (x == 0),
    group2_filter=lambda x: (x == 1), 
    group1_name='Healthy',
    group2_name='Diabetes Not Controlled',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='TAU_217',
    fig_dir=FIG_DIR
)

compare_groups_statistical(
    df=df, 
    group_col='Diabetes_controlled', 
    group1_filter=lambda x: (x == 0),
    group2_filter=lambda x: (x == 1), 
    group1_name='Healthy',
    group2_name='Diabetes Not Controlled',
    test_type='mannwhitneyu',
    alternative='greater',
    fig_dir=FIG_DIR
)

compare_groups_statistical(
    df=df, 
    group_col='Diabetes_controlled', 
    feature_col=GFAP,
    group1_filter=lambda x: (x == 0),
    group2_filter=lambda x: (x == 1), 
    group1_name='Healthy',
    group2_name='Diabetes Not Controlled',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='GFAP',
    fig_dir=FIG_DIR
)

compare_groups_statistical(
    df=df, 
    group_col='Diabetes_controlled', 
    feature_col=NFL,
    group1_filter=lambda x: (x == 0),
    group2_filter=lambda x: (x == 1), 
    group1_name='Healthy',
    group2_name='Diabetes Not Controlled',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='NFL',
    fig_dir=FIG_DIR
)

In [ ]:
# same but filtering out subject with 5.8 < Hemoglobin_A1C < 6.5
diabetes_not_controled = df[(df['Hemoglobin_A1C'] >= 6.5)]
diabetes_controled = df[(df['Hemoglobin_A1C'] < 6.5)]
df['Diabetes_controlled'] = np.nan
df['Diabetes_controlled'][(df['Hemoglobin_A1C'] < 5.8)] = 0
df['Diabetes_controlled'][(df['Hemoglobin_A1C'] >= 6.5)] = 1
## remove nan in diabetes_controlled
# df_diabetes = df.dropna(subset='Diabetes_controlled')


# compare_groups_statistical(
#     df=df_diabetes, 
#     group_col='Diabetes_controlled', 
#     feature_col=TAU_217,
#     group1_filter=lambda x: (x == 0),
#     group2_filter=lambda x: (x == 1), 
#     group1_name='Healthy',
#     group2_name='Diabetes Not Controlled',
#     test_type='mannwhitneyu',
#     alternative='greater',
#     feature_name='TAU_217',
#     fig_dir=FIG_DIR
# )

# compare_groups_statistical(
#     df=df_diabetes, 
#     group_col='Diabetes_controlled', 
#     group1_filter=lambda x: (x == 0),
#     group2_filter=lambda x: (x == 1), 
#     group1_name='Healthy',
#     group2_name='Diabetes Not Controlled',
#     test_type='mannwhitneyu',
#     alternative='greater',
#     fig_dir=FIG_DIR
# )

# compare_groups_statistical(
#     df=df_diabetes, 
#     group_col='Diabetes_controlled', 
#     feature_col=GFAP,
#     group1_filter=lambda x: (x == 0),
#     group2_filter=lambda x: (x == 1), 
#     group1_name='Healthy',
#     group2_name='Diabetes Not Controlled',
#     test_type='mannwhitneyu',
#     alternative='greater',
#     feature_name='GFAP',
#     fig_dir=FIG_DIR
# )

# compare_groups_statistical(
#     df=df_diabetes, 
#     group_col='Diabetes_controlled', 
#     feature_col=NFL,
#     group1_filter=lambda x: (x == 0),
#     group2_filter=lambda x: (x == 1), 
#     group1_name='Healthy',
#     group2_name='Diabetes Not Controlled',
#     test_type='mannwhitneyu',
#     alternative='greater',
#     feature_name='NFL',
#     fig_dir=FIG_DIR
# )

In [ ]:
diabetes_not_controled = df[(df['Diabetes'] == 'YES') & (df['Hemoglobin_A1C'] >= 6.7)]
diabetes_controled = df[(df['Diabetes'] == 'YES') & (df['Hemoglobin_A1C'] < 6.7)]
all_diabetes = df[df['Diabetes'] == 'YES']
only_healthy = df[df['Diabetes'] == 'NO']
alpha = 0.2

sns.histplot(data=only_healthy, x=TAU_217, kde=True, stat='density', element='step', label=f'Only healthy subjects: {only_healthy.shape[0]}', alpha=alpha)
sns.histplot(data=all_diabetes, x=TAU_217, kde=True, stat='density', element='step', label=f'All subjects with diabetes: {all_diabetes.shape[0]}', alpha=alpha)
sns.histplot(data=diabetes_controled, x=TAU_217, kde=True, stat='density', element='step', label=f'Controled: {diabetes_controled.shape[0]}', alpha=alpha)
sns.histplot(data=diabetes_not_controled, x=TAU_217, kde=True, stat='density', element='step', label=f'Not controled: {diabetes_not_controled.shape[0]}', alpha=alpha)
plt.legend()
plt.xlabel('Tau 217')
plt.title('Diabetes Controled vs Not Controled')
# plt.savefig(SAVE_DIR + 'Diabetes_Controled_vs_Not_Controled.pdf', bbox_inches='tight', dpi=300, format='pdf')
plt.show()

plt.legend()
plt.title('Diabetes Controled vs Not Controled')
plt.show()

# GFAP
sns.histplot(data=only_healthy, x=GFAP, kde=True, stat='density', element='step', label=f'Only healthy subjects: {only_healthy.shape[0]}', alpha=alpha)
sns.histplot(data=all_diabetes, x=GFAP, kde=True, stat='density', element='step', label=f'All subjects with diabetes: {all_diabetes.shape[0]}', alpha=alpha)
sns.histplot(data=diabetes_controled, x=GFAP, kde=True, stat='density', element='step', label=f'Controled: {diabetes_controled.shape[0]}', alpha=alpha)
sns.histplot(data=diabetes_not_controled, x=GFAP, kde=True, stat='density', element='step', label=f'Not controled: {diabetes_not_controled.shape[0]}', alpha=alpha)
plt.legend()
plt.xlabel('GFAP')
plt.title('Diabetes Controled vs Not Controled')
plt.show()
#NFL
sns.histplot(data=only_healthy, x=NFL, kde=True, stat='density', element='step', label=f'Only healthy subjects: {only_healthy.shape[0]}', alpha=alpha)
sns.histplot(data=all_diabetes, x=NFL, kde=True, stat='density', element='step', label=f'All subjects with diabetes: {all_diabetes.shape[0]}', alpha=alpha)
sns.histplot(data=diabetes_controled, x=NFL, kde=True, stat='density', element='step', label=f'Controled: {diabetes_controled.shape[0]}', alpha=alpha)
sns.histplot(data=diabetes_not_controled, x=NFL, kde=True, stat='density', element='step', label=f'Not controled: {diabetes_not_controled.shape[0]}', alpha=alpha)
plt.legend()
plt.xlabel('NFL')
plt.title('Diabetes Controled vs Not Controled')
plt.show()

### Treatd risk factor
We want to check for each of the risk factors that can be tested using blood test if they are indeed controlled or not  

Diabetes melitus

In [ ]:
not_controlled, controlled, healthy, reported, all_cohort = 40, 73, 530, 136, 530 + 136
print(f"diabetes not controlled out of the full cohort: {not_controlled/all_cohort}")
print(f"diabetes not controlled out of all the reported cases: {not_controlled/reported}") 

CRP

In [ ]:
df_crp = df_217[df_217['CRP'].notna()]
df_crp['CRP'].replace({"<0.5":0, "<1":0, "<0.4":0, "<0.6":0}, inplace=True)
df_crp['CRP'] = df_crp["CRP"].astype(float)
crp_cutoff = 1
df_crp["CRP_not_controlled"] = df["CRP"] > crp_cutoff
df_crp['CRP_not_controlled'].value_counts()

HDL

In [ ]:
df_hdl = df_217[df_217['HDL'].notna()]
df_hdl['HDL'] = df_hdl["HDL"].astype(float)
hdl_cutoff = 60
df_hdl["HDL_not_controlled"] = df_hdl["HDL"] < hdl_cutoff
df_hdl['HDL_not_controlled'].value_counts()
(df_hdl['HDL_not_controlled'][df_hdl['Hyperlipidemia'] == "YES"]).value_counts()

LDL

In [ ]:
df_ldl = df_217[df_217['LDL'].notna()]
df_ldl['LDL'] = df_ldl["LDL"].astype(float)
ldl_cutoff = 100
df_ldl["LDL_not_controlled"] = df_ldl["LDL"] >= ldl_cutoff
print(df_ldl['LDL_not_controlled'].value_counts())
(df_ldl['LDL_not_controlled'][df_ldl['Hyperlipidemia'] == "YES"]).value_counts()
df_ldl['HDL_LDL_not_controlled'] = (df_ldl['LDL_not_controlled'] & df_hdl['HDL_not_controlled'])
(df_ldl['HDL_LDL_not_controlled'][df_ldl['Hyperlipidemia'] == "YES"]).value_counts()

In [ ]:
hyperlipedimia_not_controlled = df_ldl[(df_ldl['HDL_LDL_not_controlled'] == True)]
hyperlipedimia_controlled = df_ldl[(df_ldl['HDL_LDL_not_controlled'] == False)]

all_hyperlipedimia = df_ldl[df_ldl['Hyperlipidemia'] == 'YES']
only_healthy = df_ldl[df_ldl['Hyperlipidemia'] == 'NO']
alpha = 0.2

compare_groups_statistical(
    df=df_ldl, 
    group_col='HDL_LDL_not_controlled', 
    feature_col=TAU_217,
    group1_filter=lambda x: (x == 0),
    group2_filter=lambda x: (x == 1), 
    group1_name='Healthy',
    group2_name='Hyperlipidemia Not Controlled',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='TAU_217',
    fig_dir=FIG_DIR
)

compare_groups_statistical(
    df=df_ldl, 
    group_col='HDL_LDL_not_controlled', 
    group1_filter=lambda x: (x == 0),
    group2_filter=lambda x: (x == 1), 
    group1_name='Healthy',
    group2_name='Hyperlipidemia Not Controlled',
    test_type='mannwhitneyu',
    alternative='greater',
    fig_dir=FIG_DIR
)

compare_groups_statistical(
    df=df_ldl, 
    group_col='HDL_LDL_not_controlled', 
    feature_col=GFAP,
    group1_filter=lambda x: (x == 0),
    group2_filter=lambda x: (x == 1), 
    group1_name='Healthy',
    group2_name='Hyperlipidemia Not Controlled',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='GFAP',
    fig_dir=FIG_DIR
)

compare_groups_statistical(
    df=df_ldl, 
    group_col='HDL_LDL_not_controlled', 
    feature_col=NFL,
    group1_filter=lambda x: (x == 0),
    group2_filter=lambda x: (x == 1), 
    group1_name='Healthy',
    group2_name='Hyperlipidemia Not Controlled',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='NFL',
    fig_dir=FIG_DIR
)

Cholesterol

In [ ]:
df_chol = df_217[df_217['Cholesterol'].notna()]
df_chol['Cholesterol'] = df_chol["Cholesterol"].astype(float)
chol_cutoff = 200
df_chol["Cholesterol_not_controlled"] = df_chol["Cholesterol"] > chol_cutoff
print(df_chol['Cholesterol_not_controlled'].value_counts())
# (df_chol['Cholesterol_not_controlled'][df_chol['Hyperlipidemia'] == "YES"]).value_counts()
df_ldl['HDL_LDL_not_controlled'][(df_chol['Cholesterol_not_controlled'] & (df_ldl["Hyperlipidemia"] == "YES"))].value_counts()

In [ ]:
cholesterol_not_controlled = df_chol[(df_chol["Cholesterol_not_controlled"] == True)]
cholesterol_controlled = df_chol[(df_chol["Cholesterol_not_controlled"] == False)]

all_hyperlipedimia = df_chol[df_chol['Hyperlipidemia'] == 'YES']
only_healthy = df_ldl[df_ldl['Hyperlipidemia'] == 'NO']

compare_groups_statistical(
    df=df_chol, 
    group_col='Cholesterol_not_controlled', 
    feature_col=TAU_217,
    group1_filter=lambda x: (x == 0),
    group2_filter=lambda x: (x == 1), 
    group1_name='Healthy',
    group2_name='Cholesterol Not Controlled',
    test_type='mannwhitneyu',
    alternative='less',
    feature_name='TAU_217',
    fig_dir=FIG_DIR
)

compare_groups_statistical(
    df=df_chol, 
    group_col='Cholesterol_not_controlled', 
    group1_filter=lambda x: (x == 0),
    group2_filter=lambda x: (x == 1), 
    group1_name='Healthy',
    group2_name='Cholesterol Not Controlled',
    test_type='mannwhitneyu',
    alternative='less',
    fig_dir=FIG_DIR
)

compare_groups_statistical(
    df=df_chol, 
    group_col='Cholesterol_not_controlled', 
    feature_col=GFAP,
    group1_filter=lambda x: (x == 0),
    group2_filter=lambda x: (x == 1), 
    group1_name='Healthy',
    group2_name='Cholesterol Not Controlled',
    test_type='mannwhitneyu',
    alternative='less',
    feature_name='GFAP',
    fig_dir=FIG_DIR
)

compare_groups_statistical(
    df=df_chol, 
    group_col='Cholesterol_not_controlled', 
    feature_col=NFL,
    group1_filter=lambda x: (x == 0),
    group2_filter=lambda x: (x == 1), 
    group1_name='Healthy',
    group2_name='Cholesterol Not Controlled',
    test_type='mannwhitneyu',
    alternative='less',
    feature_name='NFL',
    fig_dir=FIG_DIR
)

Cholesterol/HDL

In [ ]:
df_217["Cholesterol/HDL"] = df_217['Cholesterol'] / df_217['HDL']
print(df_217["Cholesterol/HDL"].notna().sum())

df_217["Cholesterol/HDL_not_controlled"] = df_217["Cholesterol/HDL"] > 5
df_217['Cholesterol/HDL'].describe(), df_217["Cholesterol/HDL_not_controlled"].value_counts()

### Does females with high tau217 and high GFAP have a higher risk of dementia?

In [ ]:
high_tau_217 = df[df[TAU_217] > HIGH_TAU_217_THR]
# scatter plot of gfap vs MOCA_score
fig, ax = plt.subplots(figsize=(10, 5))
sns.scatterplot(data=df, x=GFAP, y=MOCA_COL, ax=ax, hue=TAU_217)
# ax.set_ylim(30, 500)
ax.set_title('GFAP vs MOCA_score')
ax.set_xlabel('GFAP')
ax.set_ylabel('MOCA_score')
plt.show()



In [ ]:
low_tau_217 = df[df[TAU_217] <= HIGH_TAU_217_THR]
high_tau_217 = df[df[TAU_217] > HIGH_TAU_217_THR]
high_gfap = df[df[GFAP] > HIGH_GFAP_THR]
low_tau_217_and_high_gfap = low_tau_217[low_tau_217[GFAP] > HIGH_GFAP_THR]
print(low_tau_217_and_high_gfap.shape)
low_tau_217_and_low_gfap = low_tau_217[low_tau_217[GFAP] <= HIGH_GFAP_THR] 
high_tau_217_and_high_gfap = high_tau_217[high_tau_217[GFAP] > HIGH_GFAP_THR]
print(high_tau_217_and_high_gfap.shape)

fig, ax = plt.subplots(figsize=(10, 5))
sns.scatterplot(data=high_tau_217_and_high_gfap, x=TAU_217, y=MOCA_COL, hue='Gender')
# sns.scatterplot(data=high_tau_217_and_high_gfap, x=TAU_217, y=MOCA_COL, hue='Gender')

# Calculate the correlation MOCA and gender in the high tau217 and high gfap
male_corr = spearmanr(
    high_tau_217[high_tau_217['Gender'] == 'm'][MOCA_COL].astype(float),
    high_tau_217[high_tau_217['Gender'] == 'm'][GFAP]
)
print(male_corr)

female_corr = spearmanr(
    high_tau_217[high_tau_217['Gender'] == 'f'][MOCA_COL].astype(float),
    high_tau_217[high_tau_217['Gender'] == 'f'][GFAP]
)
print(female_corr)



# sns.scatterplot(data=high_gfap, x=TAU_217, y=TMT_B_COL, hue=MCI)
# ax.set_ylim(10, 30)

plt.show()

## OSA controlled vs not

In [ ]:
df_217.columns

In [ ]:
df_217['severity of OSA'].value_counts()
df_217['severe OSA'] 

# Biomarkers correlation

In [ ]:
# plot tau217 vs gfap

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

sns.scatterplot(data=scaled_b_df, x=TAU_217, y='GFAP', ax=ax1)
ax1.set_title('Tau 217 vs GFAP')
ax1.vlines(x=SCALED_HIGH_TAU_217_THR, ymin=-0.01, ymax=1, color='r', linestyle='--', label='Scaled High \nTau 217 Threshold')
# ax1.legend(labels=['Healthy', 'MCI', 'Scaled High \nTau 217 Threshold'])

sns.scatterplot(data=scaled_b_df[scaled_b_df[TAU_217] > SCALED_HIGH_TAU_217_THR], x=TAU_217, y='GFAP', ax=ax2)
ax2.vlines(x=SCALED_HIGH_TAU_217_THR, ymin=-0.01, ymax=1, color='r', linestyle='--', label='Scaled High \nTau 217 Threshold')
# calculate correlation between tau217 and gfap
corr = spearmanr(scaled_b_df[TAU_217], scaled_b_df['GFAP'])
print(corr)
corr = spearmanr(scaled_b_df[scaled_b_df[TAU_217] > SCALED_HIGH_TAU_217_THR][TAU_217], scaled_b_df[scaled_b_df[TAU_217] > SCALED_HIGH_TAU_217_THR]['GFAP'])
print(corr)

In [ ]:
compare_groups_statistical(
    df=df, 
    group_col='Gender', 
    feature_col=NFL,
    group1_filter=lambda x: (x == 'm'),  # APOE3/3
    group2_filter=lambda x: (x == 'f'),  # APOE3/4 or APOE4/4
    group1_name='Male',
    group2_name='Female',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='NFL',
    fig_dir=FIG_DIR
)

In [ ]:
compare_groups_statistical(
    df=df[df[TAU_217_CAT] == 3], 
    group_col='Gender', 
    feature_col=NFL,
    group1_filter=lambda x: (x == 'm'),  # APOE3/3
    group2_filter=lambda x: (x == 'f'),  # APOE3/4 or APOE4/4
    group1_name='Male',
    group2_name='Female',
    test_type='mannwhitneyu',
    alternative='greater',
    feature_name='NFL',
    fig_dir=FIG_DIR
)

In [ ]:
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# sns.scatterplot(data=scaled_b_df, x=TAU_217, y='NFL', hue='MCI', ax=ax1)
# ax1.set_title('Tau 217 vs NFL')
# ax1.vlines(x=SCALED_HIGH_TAU_217_THR, ymin=-0.01, ymax=1, color='r', linestyle='--', label='Scaled High \nTau 217 Threshold')
# ax1.legend(labels=['Healthy', 'MCI', 'Scaled High \nTau 217 Threshold'])

# ax2.legend(labels=['Healthy', 'MCI', 'Scaled High \nTau 217 Threshold'])

# plt.suptitle('Tau vs NFL', fontsize=20)
# # plt.savefig('final_figures/tau_vs_NFL.pdf')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

sns.scatterplot(data=scaled_b_df, x=TAU_217, y='GFAP', hue='MCI', ax=ax1)
ax1.set_title('Tau 217 vs GFAP')
ax1.vlines(x=SCALED_HIGH_TAU_217_THR, ymin=-0.01, ymax=1, color='r', linestyle='--', label='Scaled High \nTau 217 Threshold')
ax1.legend(labels=['Healthy', 'MCI', 'Scaled High \nTau 217 Threshold'])

# ax2.legend(labels=['Healthy', 'MCI', 'Scaled High \nTau 217 Threshold'])

# plt.suptitle('Tau vs GFAP', fontsize=20)
# plt.savefig('final_figures/tau_vs_GFAP.pdf')

In [ ]:
df_217 = df[df[TAU_217].notna()]

sns.heatmap(corr_217, annot=True, vmin=0, vmax=1)
plt.title(f'Correlation heatmap\n all p-tau217 samples N={df_217.shape[0]}')
plt.savefig(f'{FIG_DIR}/Correlation heatmap, all p-tau217 samples.pdf')

# Check if High GFAP individuals are high on HDL and low on p-tau217

In [ ]:
INT_GFAP_THR

In [ ]:
def cat_gfap(df):
    gfap_cat = np.ones(df.shape[0])
    gfap_cat[(HIGH_GFAP_THR >= df['GFAP']) & (df['GFAP'] >= INT_GFAP_THR)] = 2
    gfap_cat[(df['GFAP'] >= HIGH_GFAP_THR)] = 3
    df['GFAP_CAT'] = gfap_cat
    return df


In [ ]:
df[df[TAU_217_CAT] == 3][['GFAP', 'HDL', 'p-tau217', MOCA_COL]].corr(method='spearman')

In [ ]:
# Check if High GFAP individuals are high on HDL and low on p-tau217
df = cat_gfap(df)
df[['GFAP', 'GFAP_CAT', 'HDL', 'p-tau217']]
df_not_low_gfap = df[df['GFAP_CAT'] > 1]
df_not_low_gfap[['GFAP', 'GFAP_CAT', 'HDL', 'p-tau217']].corr(method='spearman')
df_high_gfap = df[df['GFAP_CAT'] == 3]
corr = df_high_gfap[[GFAP, TAU_217, NFL, 'HDL', MOCA_COL]].corr(method='spearman')
# plot correlation heatmap
sns.heatmap(corr, annot=True, vmin=-1, vmax=1, cmap='coolwarm')
plt.title('Spearman correlation heatmap of biomarkers in high GFAP individuals')
plt.savefig(f'{FIG_DIR}/Spearman correlation heatmap of biomarkers in high GFAP individuals.pdf')
plt.show()



In [ ]:
# plot the high gfap individuals p-tau vs HDL and color by MOCA_score
fig, ax = plt.subplots(figsize=(10, 5))
sns.scatterplot(data=df_high_gfap.dropna(subset=['p-tau217', 'HDL', MOCA_COL]), x='p-tau217', y='HDL', hue=MCI, ax=ax, palette='coolwarm')
ax.set_title('p-tau217 vs HDL in high GFAP individuals')
ax.set_xlabel('p-tau217')
ax.set_ylabel('HDL')
plt.show()

In [ ]:
# plot the high gfap individuals p-tau vs MOCA score and color by HDL
fig, ax = plt.subplots(figsize=(10, 5))
sns.scatterplot(data=df_high_gfap.dropna(subset=['p-tau217', 'HDL', MOCA_COL]), x='p-tau217', y=MOCA_COL, hue='HDL', ax=ax, palette='coolwarm')
ax.set_title('p-tau217 vs HDL in high GFAP individuals')
ax.set_xlabel('p-tau217')
ax.set_ylabel('MOCA score')
plt.show()


NFL

In [ ]:
def get_nfl_threshold(age):
    """Get NFL threshold based on age group"""
    if age < 70:
        return HIGH_NFL_65_69_THR
    elif age < 75:
        return HIGH_NFL_70_74_THR
    elif age >= 75:
        return HIGH_NFL_75_80_THR

def cat_nfl(df):
    nfl_cat = np.ones(df.shape[0])
    nfl_cat[df[NFL] > df['Age'].apply(get_nfl_threshold)] = 2
    df['NFL_CAT'] = nfl_cat
    return df

df = cat_nfl(df)


In [ ]:
df_high_nfl = df[df['NFL_CAT'] == 2]
df_high_nfl[['NFL', 'GFAP', 'HDL', 'p-tau217', MOCA_COL]].corr(method='spearman')

# Check if biomarkers level is effect by kidny function by looking on GFR and Creatinine

In [ ]:
kidney_df = df[['GFR', 'Creatinine'] + ['p-tau217', 'GFAP', 'NFL'] + [MOCA_COL] + ['ID']]
kidney_df.dropna(inplace=True)
print(kidney_df.shape)
# remove the top 3 samples with highest Creatinine and print their ids
kidney_df = kidney_df.sort_values(by='Creatinine', ascending=False)
print(kidney_df.head(3))
kidney_df = kidney_df.iloc[3:].drop(columns=['ID'])
print(kidney_df.shape)
#plot heatmap of kidney_df spearman correlation
sns.heatmap(kidney_df.corr(method='spearman'), annot=True, vmin=-1, vmax=1)
plt.title("spearman correlation")
plt.show()

sns.heatmap(kidney_df.corr(method='pearson'), annot=True, vmin=-1, vmax=1)
plt.title("pearson correlation")
plt.show()

In [ ]:
# Plot creatinine vs all biomarkers in one figure with 3 subplots
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)

# p-tau217 plot
sns.scatterplot(data=kidney_df, x='Creatinine', y='p-tau217', color='blue', ax=axes[0])
axes[0].axhline(y=0.444, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Creatinine vs p-tau217')
axes[0].set_xlabel('Creatinine')
axes[0].set_ylabel('p-tau217')

# NFL plot
sns.scatterplot(data=kidney_df, x='Creatinine', y='NFL', color='red', ax=axes[1])
axes[1].set_title('Creatinine vs NFL')
axes[1].set_xlabel('Creatinine')
axes[1].set_ylabel('NFL')

# GFAP plot
sns.scatterplot(data=kidney_df, x='Creatinine', y='GFAP', color='green', ax=axes[2])
axes[2].set_title('Creatinine vs GFAP')
axes[2].set_xlabel('Creatinine')
axes[2].set_ylabel('GFAP')

plt.tight_layout()
plt.show()

In [ ]:
# Same but for GFR
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)
# p-tau217 plot
sns.scatterplot(data=kidney_df, x='GFR', y='p-tau217', color='blue', ax=axes[0])
axes[0].set_title('GFR vs p-tau217')
axes[0].axhline(y=0.444, color='black', linestyle='--', linewidth=1)
axes[0].set_xlabel('GFR')
axes[0].set_ylabel('p-tau217')

# NFL plot
sns.scatterplot(data=kidney_df, x='GFR', y='NFL', color='red', ax=axes[1])
axes[1].set_title('GFR vs NFL')
axes[1].set_xlabel('GFR')
axes[1].set_ylabel('NFL')

# GFAP plot
sns.scatterplot(data=kidney_df, x='GFR', y='GFAP', color='green', ax=axes[2])
axes[2].set_title('GFR vs GFAP')
axes[2].set_xlabel('GFR')
axes[2].set_ylabel('GFAP')

plt.tight_layout()
plt.show()

# Looking into the correlation of biomarkers with MBI-C

In [ ]:
behavioral_df = df_217[['p-tau217', 'GFAP', 'NFL'] + [MOCA_COL, MBI_C_COL, 'following societal norms'] + ['ID']]
behavioral_df.dropna(inplace=True)
print(behavioral_df.shape)

print(behavioral_df.head(3))
behavioral_df = behavioral_df.iloc[3:].drop(columns=['ID'])
print(behavioral_df.shape)
#plot heatmap of behavioral_df spearman correlation
sns.heatmap(behavioral_df.corr(method='spearman'), annot=True, vmin=-1, vmax=1)
plt.title("spearman correlation")
plt.savefig(f'{FIG_DIR}/Spearman Correlation heatmap, behavioral with biomarkers.pdf')
plt.show()

sns.heatmap(behavioral_df.corr(method='pearson'), annot=True, vmin=-1, vmax=1)
plt.title("pearson correlation")
plt.savefig(f'{FIG_DIR}/Pearson Correlation heatmap, behavioral with biomarkers.pdf')
plt.show()